# QUY TRÌNH 2 - DATA INGESTION

## E-commerce Data Lakehouse trên Object Storage

Mục tiêu:

- Đọc dữ liệu CSV Raw từ Quy trình 1
- Đọc dữ liệu ảnh sản phẩm từ thư mục images
- Kiểm tra metadata của dữ liệu
- Kết nối đến MinIO
- Upload dữ liệu bảng vào Object Storage
- Upload dữ liệu ảnh vào Object Storage
- Ghi log quá trình ingestion
- Kiểm tra kết quả ingestion

In [1]:
import os
import json
from pathlib import Path
from datetime import datetime, timezone

import pandas as pd
from minio import Minio

In [2]:
# Thiết lập đường dẫn project

BASE_DIR = Path.cwd().resolve()

if BASE_DIR.name == "notebooks":
    BASE_DIR = BASE_DIR.parent

RAW_DIR = BASE_DIR / "data" / "raw"
IMAGE_DIR = BASE_DIR / "data" / "images" / "products"

# Kiểm tra thư mục CSV Raw
if not RAW_DIR.exists():
    raise FileNotFoundError(
        f"Không tìm thấy Raw directory: {RAW_DIR}"
    )

# Kiểm tra thư mục ảnh
if not IMAGE_DIR.exists():
    raise FileNotFoundError(
        f"Không tìm thấy Image directory: {IMAGE_DIR}"
    )

print("Project    :", BASE_DIR)
print("Raw CSV    :", RAW_DIR)
print("Image data :", IMAGE_DIR)

Project    : /Users/camlee05/development/E-commerce-Data-Lakehouse
Raw CSV    : /Users/camlee05/development/E-commerce-Data-Lakehouse/data/raw
Image data : /Users/camlee05/development/E-commerce-Data-Lakehouse/data/images/products


In [3]:
# Cấu hình kết nối MinIO
MINIO_ENDPOINT = "localhost:9000"
MINIO_ACCESS_KEY = os.getenv("MINIO_ROOT_USER", "minioadmin")
MINIO_SECRET_KEY = os.getenv("MINIO_ROOT_PASSWORD", "minioadmin")

client = Minio(
    MINIO_ENDPOINT,
    access_key=MINIO_ACCESS_KEY,
    secret_key=MINIO_SECRET_KEY,
    secure=False
)

print("Đã khởi tạo MinIO client.")

Đã khởi tạo MinIO client.


In [4]:
# Kiểm tra kết nối MinIO và liệt kê các bucket
# Lưu ý: Khởi động MinIO trước khi chạy cell này.
# Chạy tại thư mục gốc của project:
#     docker-compose up -d
# Sau đó kiểm tra MinIO Console tại:
#     http://localhost:9001
# Kiểm tra kết nối MinIO

try:
    buckets = list(client.list_buckets())

    print("✓ Kết nối MinIO thành công")

    if buckets:
        print("Các bucket hiện có:")
        for bucket in buckets:
            print("-", bucket.name)
    else:
        print("Chưa có bucket nào.")

except Exception as e:
    raise ConnectionError(
        f"Không thể kết nối MinIO tại "
        f"{MINIO_ENDPOINT}: {e}"
    )

✓ Kết nối MinIO thành công
Các bucket hiện có:
- ecommerce-lakehouse


In [5]:
# Tạo bucket 
BUCKET_NAME = "ecommerce-lakehouse"

if not client.bucket_exists(BUCKET_NAME):
    client.make_bucket(BUCKET_NAME)
    print(f"Đã tạo bucket: {BUCKET_NAME}")
else:
    print(f"Bucket đã tồn tại: {BUCKET_NAME}")

Bucket đã tồn tại: ecommerce-lakehouse


In [6]:
# Khai báo các CSV Raw

DATASETS = {
    "customers": RAW_DIR / "customers.csv",
    "products": RAW_DIR / "products.csv",
    "orders": RAW_DIR / "orders.csv",
    "order_items": RAW_DIR / "order_items.csv",
    "reviews": RAW_DIR / "reviews.csv",
    "payments": RAW_DIR / "payments.csv"
}

# Kiểm tra file
missing_files = [
    str(path)
    for path in DATASETS.values()
    if not path.exists()
]

if missing_files:
    raise FileNotFoundError(
        f"Thiếu các Raw CSV: {missing_files}"
    )

# Đọc dữ liệu
datasets = {}

for name, path in DATASETS.items():

    datasets[name] = pd.read_csv(path)

    print(
        f"✓ {name}: "
        f"{len(datasets[name]):,} records"
    )

✓ customers: 99,441 records
✓ products: 32,951 records
✓ orders: 99,441 records
✓ order_items: 112,650 records
✓ reviews: 99,224 records
✓ payments: 103,886 records


In [7]:
# Thu thập danh sách ảnh sản phẩm

SUPPORTED_IMAGE_EXTENSIONS = {
    ".jpg",
    ".jpeg",
    ".png",
    ".webp"
}

IMAGE_FILES = [
    file
    for file in IMAGE_DIR.iterdir()
    if file.is_file()
    and file.suffix.lower()
    in SUPPORTED_IMAGE_EXTENSIONS
]

print(
    f"✓ Tìm thấy {len(IMAGE_FILES):,} ảnh sản phẩm"
)

✓ Tìm thấy 5 ảnh sản phẩm


In [8]:
# Thu thập metadata của CSV

def get_file_metadata(file_path):

    stat = file_path.stat()

    return {
        "file_name": file_path.name,
        "file_type": file_path.suffix.lower(),
        "file_size_bytes": stat.st_size,
        "file_size_mb": round(
            stat.st_size / (1024 * 1024),
            2
        ),
        "modified_time": datetime.fromtimestamp(
            stat.st_mtime,
            tz=timezone.utc
        ).isoformat()
    }


metadata = []

for name, path in DATASETS.items():

    info = get_file_metadata(path)

    info["dataset"] = name
    info["rows"] = len(datasets[name])
    info["columns"] = len(
        datasets[name].columns
    )

    metadata.append(info)


metadata_df = pd.DataFrame(metadata)

metadata_df

,file_name,file_type,file_size_bytes,file_size_mb,modified_time,dataset,rows,columns
0,customers.csv,.csv,8562270,8.17,2026-09-25T10:20:38.047406+00:00,customers,99441,5
1,products.csv,.csv,2795930,2.67,2026-09-25T10:20:38.110873+00:00,products,32951,9
2,orders.csv,.csv,17406884,16.60,2026-09-25T10:20:38.284757+00:00,orders,99441,8
3,order_items.csv,.csv,14911576,14.22,2026-09-25T10:20:38.471162+00:00,order_items,112650,7
4,reviews.csv,.csv,13955535,13.31,2026-09-25T10:20:38.621352+00:00,reviews,99224,7
5,payments.csv,.csv,5633362,5.37,2026-09-25T10:20:38.707670+00:00,payments,103886,5


In [9]:
# Thu thập metadata ảnh

image_metadata = []

for image_path in IMAGE_FILES:

    stat = image_path.stat()

    image_metadata.append({
        "file_name": image_path.name,
        "file_type": image_path.suffix.lower(),
        "file_size_bytes": stat.st_size,
        "file_size_kb": round(
            stat.st_size / 1024,
            2
        ),
        "modified_time": datetime.fromtimestamp(
            stat.st_mtime,
            tz=timezone.utc
        ).isoformat()
    })

image_metadata_df = pd.DataFrame(
    image_metadata
)

image_metadata_df.head()

,file_name,file_type,file_size_bytes,file_size_kb,modified_time
0,product_005.jpg,.jpg,126543,123.58,2026-09-24T16:08:18.003860+00:00
1,product_004.jpg,.jpg,36381,35.53,2026-09-24T16:07:53.108337+00:00
2,product_001.jpg,.jpg,181972,177.71,2026-09-24T16:06:15.397657+00:00
3,product_003.jpg,.jpg,41352,40.38,2026-09-24T16:07:10.121806+00:00
4,product_002.jpg,.jpg,229085,223.72,2026-09-24T16:06:46.734289+00:00


In [10]:
# Kiểm tra schema của các dataset
schema_info = []

for name, df in datasets.items():
    for column in df.columns:
        schema_info.append({
            "dataset": name,
            "column": column,
            "dtype": str(df[column].dtype)
        })

schema_df = pd.DataFrame(schema_info)

schema_df.head(20)

,dataset,column,dtype
0,customers,customer_id,str
1,customers,customer_unique_id,str
2,customers,customer_zip_code_prefix,int64
3,customers,customer_city,str
4,customers,customer_state,str
5,products,product_id,str
6,products,product_category_name,str
7,products,product_name_lenght,float64
8,products,product_description_lenght,float64
9,products,product_photos_qty,float64


In [11]:
# Log toàn bộ quá trình ingestion
ingestion_logs = []

# Upload CSV Raw lên MinIO

for name, path in DATASETS.items():

    object_name = f"raw/{path.name}"

    try:

        file_size = path.stat().st_size

        client.fput_object(
            BUCKET_NAME,
            object_name,
            str(path),
            content_type="text/csv"
        )

        ingestion_logs.append({
            "data_type": "CSV",
            "file": path.name,
            "object": object_name,
            "status": "SUCCESS",
            "size_bytes": file_size,
            "size_mb": round(
                file_size / (1024 * 1024),
                2
            ),
            "timestamp":
                datetime.now(timezone.utc).isoformat()
        })

        print(
            f"✓ CSV SUCCESS: {path.name}"
        )

    except Exception as e:

        ingestion_logs.append({
            "data_type": "CSV",
            "file": path.name,
            "object": object_name,
            "status": "FAILED",
            "timestamp":
                datetime.now(timezone.utc).isoformat(),
            "error": str(e)
        })

        print(
            f"✗ CSV FAILED: {path.name}"
        )
        print(" ", e)

✓ CSV SUCCESS: customers.csv
✓ CSV SUCCESS: products.csv
✓ CSV SUCCESS: orders.csv
✓ CSV SUCCESS: order_items.csv
✓ CSV SUCCESS: reviews.csv
✓ CSV SUCCESS: payments.csv


In [12]:
# Upload ảnh sản phẩm lên MinIO

for image_path in IMAGE_FILES:

    object_name = (
        f"images/products/{image_path.name}"
    )

    try:

        file_size = image_path.stat().st_size

        # Xác định content type
        extension = image_path.suffix.lower()

        content_types = {
            ".jpg": "image/jpeg",
            ".jpeg": "image/jpeg",
            ".png": "image/png",
            ".webp": "image/webp"
        }

        content_type = content_types.get(
            extension,
            "application/octet-stream"
        )

        client.fput_object(
            BUCKET_NAME,
            object_name,
            str(image_path),
            content_type=content_type
        )

        ingestion_logs.append({
            "data_type": "IMAGE",
            "file": image_path.name,
            "object": object_name,
            "status": "SUCCESS",
            "size_bytes": file_size,
            "size_mb": round(
                file_size / (1024 * 1024),
                2
            ),
            "timestamp":
                datetime.now(timezone.utc).isoformat()
        })

        print(
            f"✓ IMAGE SUCCESS: {image_path.name}"
        )

    except Exception as e:

        ingestion_logs.append({
            "data_type": "IMAGE",
            "file": image_path.name,
            "object": object_name,
            "status": "FAILED",
            "timestamp":
                datetime.now(timezone.utc).isoformat(),
            "error": str(e)
        })

        print(
            f"✗ IMAGE FAILED: {image_path.name}"
        )
        print(" ", e)

✓ IMAGE SUCCESS: product_005.jpg
✓ IMAGE SUCCESS: product_004.jpg
✓ IMAGE SUCCESS: product_001.jpg
✓ IMAGE SUCCESS: product_003.jpg
✓ IMAGE SUCCESS: product_002.jpg


In [13]:
# Kiểm tra CSV Raw trên MinIO

raw_objects = list(
    client.list_objects(
        BUCKET_NAME,
        prefix="raw/",
        recursive=True
    )
)

print(
    f"Số CSV object trong raw/: "
    f"{len(raw_objects)}"
)

for obj in raw_objects:
    print(
        obj.object_name,
        "|",
        obj.size,
        "bytes"
    )

Số CSV object trong raw/: 6
raw/customers.csv | 8562270 bytes
raw/order_items.csv | 14911576 bytes
raw/orders.csv | 17406884 bytes
raw/payments.csv | 5633362 bytes
raw/products.csv | 2795930 bytes
raw/reviews.csv | 13955535 bytes


In [14]:
# Kiểm tra ảnh sản phẩm trên MinIO

image_objects = list(
    client.list_objects(
        BUCKET_NAME,
        prefix="images/products/",
        recursive=True
    )
)

print(
    f"Số image object: "
    f"{len(image_objects):,}"
)

Số image object: 5


In [15]:
# Ghi log ingestion vào DataFrame
ingestion_log_df = pd.DataFrame(ingestion_logs)

ingestion_log_df

,data_type,file,object,status,size_bytes,size_mb,timestamp
0,CSV,customers.csv,raw/customers.csv,SUCCESS,8562270,8.17,2026-09-25T10:37:50.832606+00:00
1,CSV,products.csv,raw/products.csv,SUCCESS,2795930,2.67,2026-09-25T10:37:50.852957+00:00
2,CSV,orders.csv,raw/orders.csv,SUCCESS,17406884,16.60,2026-09-25T10:37:50.910705+00:00
3,CSV,order_items.csv,raw/order_items.csv,SUCCESS,14911576,14.22,2026-09-25T10:37:51.150757+00:00
4,CSV,reviews.csv,raw/reviews.csv,SUCCESS,13955535,13.31,2026-09-25T10:37:51.199409+00:00
5,CSV,payments.csv,raw/payments.csv,SUCCESS,5633362,5.37,2026-09-25T10:37:51.225187+00:00
6,IMAGE,product_005.jpg,images/products/product_005.jpg,SUCCESS,126543,0.12,2026-09-25T10:37:51.240138+00:00
7,IMAGE,product_004.jpg,images/products/product_004.jpg,SUCCESS,36381,0.03,2026-09-25T10:37:51.244749+00:00
8,IMAGE,product_001.jpg,images/products/product_001.jpg,SUCCESS,181972,0.17,2026-09-25T10:37:51.250008+00:00
9,IMAGE,product_003.jpg,images/products/product_003.jpg,SUCCESS,41352,0.04,2026-09-25T10:37:51.253207+00:00


In [16]:
# Tạo ID cho lần chạy ingestion

run_id = datetime.now(
    timezone.utc
).strftime("%Y%m%dT%H%M%SZ")


log_data = {
    "pipeline": "ecommerce-data-lakehouse",
    "process": "data_ingestion",
    "run_id": run_id,
    "timestamp":
        datetime.now(timezone.utc).isoformat(),
    "bucket": BUCKET_NAME,

    "summary": {
        "csv_files": len(DATASETS),
        "image_files": len(IMAGE_FILES)
    },

    "records": ingestion_logs
}


log_file = (
    BASE_DIR
    / "data"
    / f"ingestion_log_{run_id}.json"
)


with open(
    log_file,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        log_data,
        f,
        indent=4,
        ensure_ascii=False
    )


print(
    f"✓ Đã tạo log: {log_file}"
)

✓ Đã tạo log: /Users/camlee05/development/E-commerce-Data-Lakehouse/data/ingestion_log_20260925T103751Z.json


In [17]:
# Upload ingestion log lên MinIO

log_object_name = (
    f"logs/ingestion/"
    f"ingestion_log_{run_id}.json"
)

client.fput_object(
    BUCKET_NAME,
    log_object_name,
    str(log_file),
    content_type="application/json"
)

print(
    f"✓ Đã upload log: "
    f"{log_object_name}"
)

✓ Đã upload log: logs/ingestion/ingestion_log_20260925T103751Z.json


In [18]:
# Kiểm tra ingestion log trên MinIO

log_objects = list(
    client.list_objects(
        BUCKET_NAME,
        prefix="logs/ingestion/",
        recursive=True
    )
)

print(
    f"Số ingestion log: "
    f"{len(log_objects)}"
)

for obj in log_objects:
    print("-", obj.object_name)

Số ingestion log: 2
- logs/ingestion/ingestion_log_20260925T103519Z.json
- logs/ingestion/ingestion_log_20260925T103751Z.json


In [19]:
# Tổng kết quá trình ingestion

csv_logs = [
    log
    for log in ingestion_logs
    if log["data_type"] == "CSV"
]

image_logs = [
    log
    for log in ingestion_logs
    if log["data_type"] == "IMAGE"
]


csv_success = sum(
    log["status"] == "SUCCESS"
    for log in csv_logs
)

csv_failed = sum(
    log["status"] == "FAILED"
    for log in csv_logs
)

image_success = sum(
    log["status"] == "SUCCESS"
    for log in image_logs
)

image_failed = sum(
    log["status"] == "FAILED"
    for log in image_logs
)


pipeline_status = (
    "SUCCESS"
    if csv_failed == 0
    and image_failed == 0
    else "PARTIAL_FAILED"
)


print(
    "========== INGESTION SUMMARY =========="
)

print("Bucket:", BUCKET_NAME)

print("\n--- CSV ---")
print("Expected :", len(DATASETS))
print("Success  :", csv_success)
print("Failed   :", csv_failed)
print("MinIO    :", len(raw_objects))

print("\n--- IMAGES ---")
print("Expected :", len(IMAGE_FILES))
print("Success  :", image_success)
print("Failed   :", image_failed)
print("MinIO    :", len(image_objects))

print(
    "\nPipeline status:",
    pipeline_status
)

========== INGESTION SUMMARY ==========
Bucket: ecommerce-lakehouse

--- CSV ---
Expected : 6
Success  : 6
Failed   : 0
MinIO    : 6

--- IMAGES ---
Expected : 5
Success  : 5
Failed   : 0
MinIO    : 5

Pipeline status: SUCCESS
